# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIRˆ² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. All data entities are referenced by their `@id` in accordance with the Croissant schema and best practices.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's metadata and content with `mlcroissant`. The records, record sets, and fields are referenced by their `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published on: {metadata.datePublished}")


## 2. Data Overview
Review available record sets (`@id`), their fields (`@id`), and column details. All entity references are their complete `@id`. 

This helps understand the structure of the data for downstream analysis.

In [ ]:
# List all record sets and their fields

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    print('Available record sets:')
    for rs in metadata.recordSet:
        print(f"- Record set @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs}")
        rs_obj = dataset._metadata_object_by_id(rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs)
        # List fields
        if hasattr(rs_obj, 'field'):
            print(f"  Fields:")
            for field in rs_obj.field:
                print(f"    - field @id: {field['@id'] if isinstance(field, dict) and '@id' in field else field}")
else:
    print("No explicit recordSet entries found in metadata.recordSet. Attempting to infer from Croissant content...")
    # mlcroissant populates dataset._metadata['@graphs_final']['<DatasetID>']['recordSet'] with @id list(s); fallback:
    rec_sets = []
    if hasattr(metadata, '_metadata') and isinstance(metadata._metadata, dict):
        graphs = metadata._metadata.get('@graphs_final', {})
        for k, v in graphs.items():
            if isinstance(v, dict) and v.get('@type') == 'Dataset':
                record_sets = v.get('recordSet', [])
                rec_sets = record_sets if isinstance(record_sets, list) else [record_sets]
                break
    if not rec_sets:
        print("No record sets found in schema.")
    else:
        for rs_id in rec_sets:
            print(f"- Record set @id: {rs_id}")
            rs_obj = dataset._metadata_object_by_id(rs_id)
            if rs_obj and 'field' in rs_obj:
                print("  Fields:")
                for field in rs_obj['field']:
                    if isinstance(field, dict):
                        f_id = field.get('@id', str(field))
                    else:
                        f_id = field
                    print(f"    - field @id: {f_id}")

## 3. Data Extraction
Load the data from a specific record set into a `pandas` DataFrame for exploratory analysis. 

Refer explicitly to record set and field/column `@id`s from the previous overview.

In [ ]:
# Set the record set(s) to extract based on known Croissant schema or inspection above.
# The main table data is typically the main record set. Here we pull all record sets available.

record_sets = []
# Attempt to load record sets just as in previous cell
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        if isinstance(rs, dict) and '@id' in rs:
            record_sets.append(rs['@id'])
        else:
            record_sets.append(rs)
else:
    # Fallback same as above
    if hasattr(metadata, '_metadata') and isinstance(metadata._metadata, dict):
        graphs = metadata._metadata.get('@graphs_final', {})
        for k, v in graphs.items():
            if isinstance(v, dict) and v.get('@type') == 'Dataset':
                record_sets = v.get('recordSet', [])
                record_sets = record_sets if isinstance(record_sets, list) else [record_sets]
                break

if not record_sets:
    raise Exception("No record sets found to load data from.")

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records.")
        dataframes[record_set_id] = df
        print(f"Columns (@id): {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Failed to load records from {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group using only fields referenced by `@id` from the extracted DataFrame. Replace the example field IDs below with the actual field (usually column) `@id` as seen above for your main record set.

In [ ]:
# Choose a record set to analyze
main_record_set = record_sets[0]
df = dataframes[main_record_set]

# Display columns and choose a numeric field @id (croissant schemas often use full IRIs as keys)
print(f"Available columns in {main_record_set}:")
print(df.columns.tolist())

# Example: Let's suppose there's a field '@id': 'cr:MW' for molecular weight, and 'cr:PeptideCount' for peptide count.
# Adjust the field @id below to a numeric column present in your DataFrame.
# You can run df.columns to view real values if unknown.

numeric_field_id = None
for col in df.columns:
    # Try to pick a likely numeric field
    if 'MW' in col or 'molecular' in col.lower() or 'weight' in col.lower() or 'Count' in col or 'abundance' in col.lower():
        numeric_field_id = col
        break
# Fallback for example purposes
if not numeric_field_id:
    numeric_field_id = df.columns[0]

print(f"Using numeric field for analysis: {numeric_field_id}")

# Ensure numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].quantile(0.75)  # e.g., 75th percentile as threshold
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize this field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field if one exists
# Most biologically annotated datasets have accession or description fields.
group_field = None
for col in df.columns:
    # Look for a 'sample', 'condition', 'accession', or description field
    if 'sample' in col.lower() or 'accession' in col.lower() or 'desc' in col.lower() or 'type' in col.lower():
        group_field = col
        break

if group_field:
    print(f"Grouping filtered data by: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    display(grouped_df.head())
else:
    print("No obvious group/categorical field found for grouping analysis.")

## 5. Visualization
Let's plot the distribution of the selected numeric field and a grouped mean if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# If grouped analysis exists, show mean per group
if group_field:
    means = df.groupby(group_field)[numeric_field_id].mean().dropna()
    plt.figure(figsize=(10,4))
    means.plot(kind='bar')
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- We loaded and inspected the FAIR⁲ dataset using `mlcroissant` with all entities referenced by their Croissant schema `@id`.
- We programmatically listed fields/columns and extracted tabular data for analysis.
- After filtering and normalization on a representative numeric field, basic grouped summaries and visualizations were shown.

Further analyses can include advanced statistical testing, cross-table joins, or deeper, domain-specific investigations using the explicit structure provided by Croissant.